# Partie 2 — Nettoyage & préparation des données

**Projet CarPredictor** — suite de `01a_exploration_analyse.ipynb`.

Ce notebook applique, dans l'ordre, le traitement diagnostiqué par l'EDA :

1. Suppression des **doublons**.
2. Identification et traitement des **valeurs manquantes** (médiane pour les numériques, mode pour les catégorielles).
3. Détection des **valeurs aberrantes** (boîtes à moustaches, z‑score > 3, IQR) sur `selling_price`, `km_driven` et `year`, puis **analyse et justification** du traitement (conservation / correction / suppression).
4. **Encodage** des variables catégorielles (`fuel`, `seller_type`, `transmission`, `owner`).
5. **Séparation** train / test (80 % / 20 %).
6. **Standardisation** des variables numériques.
7. Export des jeux prêts pour la modélisation dans `data/processed/`.

Chaque étape journalise le nombre de lignes avant / après, afin de garder une trace de l'impact du nettoyage.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler



RAW_PATH = Path("../data/raw/car-price-raw-data.csv")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "selling_price"
NUM_FEATURES = ["year", "km_driven"]
CAT_FEATURES = ["fuel", "seller_type", "transmission", "owner"]
RANDOM_STATE = 42


# Exploration initiale
df = pd.read_csv(RAW_PATH)
print(f"Jeu brut : {df.shape[0]} lignes x {df.shape[1]} colonnes")
df.head()


---
## 1. Doublons

On distingue deux notions :

- le **doublon strict** : toutes les colonnes identiques (`name` compris) ;
- le **doublon fonctionnel** : mêmes caractéristiques techniques et même prix, mais libellé `name` différent.

Seuls les doublons **stricts** sont supprimés : deux annonces identiques n'apportent aucune
information nouvelle et biaisent l'apprentissage (sur‑pondération de certains profils, et
surtout risque de retrouver la même ligne dans le train **et** dans le test, ce qui
gonflerait artificiellement le score).

In [ ]:
avant = len(df)
df = df.drop_duplicates().reset_index(drop=True)

> **Décision** : suppression des **635 doublons stricts** (≈ 14,6 % du jeu).
> Les doublons « hors `name` » sont conservés : deux véhicules du même modèle, même année,
> même kilométrage arrondi et même prix sont parfaitement plausibles sur un marché
> où les annonces se ressemblent — les supprimer reviendrait à perdre de vraies observations.

---
## 2. Valeurs manquantes

Règle appliquée :

| Type | Traitement | Pourquoi |
|---|---|---|
| Cible `selling_price` | **suppression de la ligne** | une observation sans cible n'est pas exploitable en apprentissage supervisé, et l'imputer reviendrait à inventer la réponse. |
| Numériques (`year`, `km_driven`) | **imputation par la médiane** | les deux distributions sont asymétriques (§3 de l'EDA) : la médiane est insensible aux valeurs extrêmes, contrairement à la moyenne. |
| Catégorielles (`fuel`, `seller_type`, `owner`) | **imputation par le mode** | la modalité la plus fréquente est le choix le moins risqué, et les taux de manquants sont faibles (< 2,5 %). |

In [ ]:
# Imputation des valeurs manquantes
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

---
## 3. Valeurs aberrantes

Détection via boîte à moustaches, Z‑score > 3, IQR.

* `selling_price` : valeurs très élevées ou très basses suspectes.
* `km_driven` : kilomètres anormaux (ex. > 500 000).
* `year` : années futures ou antérieures à 1900.

In [ ]:
# Exemple de détection (simplifiée)
def is_outlier(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ~series.between(lower, upper)

out_price = is_outlier(df['selling_price'])
out_km = is_outlier(df['km_driven'])
out_year = is_outlier(df['year'])
outliers = out_price | out_km | out_year
print(f"Valeurs aberrantes détectées : {outliers.sum()}")

---
## 4. Encodage des variables catégorielles

On utilise `OneHotEncoder` (drop‑first) pour éviter la colinéarité.

In [ ]:
categorical_cols = ['fuel', 'seller_type', 'transmission', 'owner']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

---
## 5. Séparation train / test

80 % entraînement, 20 % test, seed = 42 pour la reproductibilité.

In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')

---
## 6. Standardisation des variables numériques

`StandardScaler` est appliqué aux colonnes numériques uniquement.

In [ ]:
numeric_features = NUM_FEATURES
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
print('Standardisation complete')

---
## 7. Export des jeux prêts pour la modélisation

Les jeux sont sauvegardés sous `data/processed/` pour être réutilisés dans les notebooks suivants.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)
print('Jeux exportés dans data/processed/')